# USDA SSURGO — Survey-Grade Soil Data (Soil Data Access API)

**What it is.** The **definitive U.S. soil database**, built by the National Cooperative
Soil Survey over a century. Far richer than global products for the U.S.: for any point you
get the soil **map unit**, its **components**, and horizon-by-horizon physical/chemical
properties.

| | |
|---|---|
| **Coverage** | United States (survey-complete for nearly all ag land) |
| **Resolution** | Field-scale soil polygons (map units) |
| **Cost / key** | **Free · no key** |
| **Access** | Soil Data Access (SDA) REST — POST a T-SQL query, get JSON |
| **Docs** | https://sdmdataaccess.sc.egov.usda.gov/WebServiceHelp.aspx |

**Why it's core to Tracera.** Soil is destiny for yield potential. **Available Water
Capacity (AWC)**, texture, drainage class, organic matter and pH drive irrigation, seeding,
and fertility decisions. This is the single most valuable *free* soil source in the U.S.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — the soil map unit at our field

In [2]:
SDA = "https://sdmdataaccess.sc.egov.usda.gov/Tabular/post.rest"

def sda_query(sql):
    """Run a Soil Data Access T-SQL query -> DataFrame (JSON+COLUMNNAME)."""
    r = requests.post(SDA, json={"query": sql, "format": "JSON+COLUMNNAME"}, timeout=90)
    r.raise_for_status()
    table = r.json().get("Table")
    if not table:
        return pd.DataFrame()
    return pd.DataFrame(table[1:], columns=table[0])

# SDA_Get_Mukey_from_intersection_with_WktWgs84 maps a lon/lat point -> soil map unit key
mu = sda_query(
    f"SELECT mu.mukey, mu.muname FROM mapunit mu WHERE mu.mukey IN "
    f"(SELECT mukey FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('point({LON} {LAT})'))"
)
assert not mu.empty, "No map unit found at point"
print("Soil map unit at the field:")
mu

Soil map unit at the field:


,mukey,muname
0,2922031,"Canisteo clay loam, Bemis moraine, 0 to 2 perc..."


### 2. Core query — horizon properties of the dominant soil component

In [3]:
soil = sda_query(f"""
    SELECT co.compname, co.comppct_r AS pct, co.drainagecl, co.taxorder,
           ch.hzname, ch.hzdept_r AS top_cm, ch.hzdepb_r AS bot_cm,
           ch.sandtotal_r AS sand_pct, ch.silttotal_r AS silt_pct, ch.claytotal_r AS clay_pct,
           ch.om_r AS organic_matter_pct, ch.ph1to1h2o_r AS pH,
           ch.awc_r AS avail_water_cap, ch.ksat_r AS ksat_um_s
    FROM component co
    JOIN chorizon ch ON co.cokey = ch.cokey
    WHERE co.mukey IN (SELECT mukey FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('point({LON} {LAT})'))
      AND co.majcompflag = 'Yes'
    ORDER BY co.comppct_r DESC, ch.hzdept_r
""")
num = ["pct","top_cm","bot_cm","sand_pct","silt_pct","clay_pct",
       "organic_matter_pct","pH","avail_water_cap","ksat_um_s"]
soil[num] = soil[num].apply(pd.to_numeric, errors="coerce")
print(f"{soil['compname'].iloc[0]} — {soil['drainagecl'].iloc[0]} ({soil['pct'].iloc[0]:.0f}% of map unit)")
soil

Canisteo — Poorly drained (85% of map unit)


,compname,pct,drainagecl,taxorder,hzname,top_cm,bot_cm,sand_pct,silt_pct,clay_pct,organic_matter_pct,pH,avail_water_cap,ksat_um_s
0,Canisteo,85,Poorly drained,Mollisols,Ap,0,20,28,43,29,7.00,7.5,0.18,2.82
1,Canisteo,85,Poorly drained,Mollisols,A,20,41,30,40,30,5.00,7.7,0.18,2.82
2,Canisteo,85,Poorly drained,Mollisols,AB,41,51,30,40,30,3.00,7.7,0.18,2.82
3,Canisteo,85,Poorly drained,Mollisols,Bkg,51,91,39,34,27,1.50,8.0,0.17,2.82
4,Canisteo,85,Poorly drained,Mollisols,Cg,91,200,49,36,15,0.25,8.0,0.18,9.17


### Notes & how to extend
- **AWC (`avail_water_cap`, cm water / cm soil)** — the water a crop can actually use; the
  key input for irrigation scheduling and drought risk. **`drainagecl`** flags tile-drainage
  needs. **`om_r`** (organic matter) and **`pH`** drive fertility.
- **Point vs field:** swap the WKT for a `polygon(...)` to get *all* map units under a field
  boundary, then area-weight the properties.
- **Bulk / offline:** for whole-county work, download gSSURGO (a file geodatabase) instead.
- **Other tables:** `muaggatt` (pre-summarized map-unit averages), `cointerp` (interpretations
  like crop suitability), `Tax...` (soil taxonomy).